# Experiment No. 06 - DCGAN CPU on Small CelebA Sample

**CPU-friendly Jupyter notebook**

Run cells from top to bottom. These notebooks are designed for an Intel i7 college PC without a dedicated GPU. Some pretrained generative models remain too slow or memory-intensive for CPU; those experiments include lightweight demonstrations and clearly marked optional full-model sections.

## Aim
Demonstrate a Deep Convolutional GAN on face images.

**CPU adaptation:** Training a DCGAN on the full CelebA dataset is not practical on a CPU-only PC. This notebook uses a small number of CelebA images and a compact network. Dataset download may require access approval.

In [ ]:
%pip install -q tensorflow tensorflow-datasets matplotlib

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

ds = tfds.load("celeb_a", split="train")
def prep(item):
    image = tf.image.resize(item["image"], (32, 32))
    return (tf.cast(image, tf.float32) - 127.5) / 127.5

train_ds = ds.map(prep).take(256).batch(16)

In [ ]:
G = tf.keras.Sequential([
    tf.keras.layers.Input((100,)),
    tf.keras.layers.Dense(4*4*64, activation="relu"),
    tf.keras.layers.Reshape((4,4,64)),
    tf.keras.layers.Conv2DTranspose(32, 4, 2, padding="same", activation="relu"),
    tf.keras.layers.Conv2DTranspose(3, 4, 2, padding="same", activation="tanh")
])
D = tf.keras.Sequential([
    tf.keras.layers.Input((16,16,3)),
    tf.keras.layers.Conv2D(32, 4, 2, padding="same", activation="relu"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1)
])
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
go = tf.keras.optimizers.Adam(0.0002)
do = tf.keras.optimizers.Adam(0.0002)

@tf.function
def train_step(real):
    n = tf.shape(real)[0]
    noise = tf.random.normal([n,100])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = G(noise, training=True)
        real_small = tf.image.resize(real, (16,16))
        real_out = D(real_small, training=True)
        fake_out = D(fake, training=True)
        gl = bce(tf.ones_like(fake_out), fake_out)
        dl = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
    gv, dv = G.trainable_variables, D.trainable_variables
    go.apply_gradients(zip(gt.gradient(gl,gv),gv))
    do.apply_gradients(zip(dt.gradient(dl,dv),dv))
    return gl, dl

for epoch in range(2):
    for real in train_ds:
        gl, dl = train_step(real)
    print("Epoch", epoch+1, "Generator loss:", float(gl), "Discriminator loss:", float(dl))

In [ ]:
faces = G(tf.random.normal([10,100]), training=False)
plt.figure(figsize=(9,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow((faces[i]+1)/2)
    plt.axis("off")
plt.show()

## Result
A compact DCGAN was trained on a small CelebA sample. Training time and output quality are limited on CPU.